# Week 7 · Task E — The note
Computes every number in `partE_note.md` from `clean/` (nothing typed by hand), then writes the half-page note: the message, the reader, each parameter with a reason, the **counterfactual** (what the raw-count map would have shown) and the **ethics clause** (what was suppressed or merged, the rule, the count).

In [1]:
import sys, os
from pathlib import Path
def _find_root():
    here = Path.cwd()
    for p in [here, here / "week7" / "assignments", here.parent / "assignments"]:
        if (p / "dashlib.py").exists() and (p / "data" / "ridership_daily.csv").exists():
            return p
    raise FileNotFoundError("Put this notebook in week7/assignments/ (next to dashlib.py) and run it from there.")
ROOT = _find_root(); os.chdir(ROOT); sys.path.insert(0, str(ROOT))
print("working in", ROOT)

working in c:\Users\Nas Graphy\Downloads\Data-Visualization\Week 7


In [2]:
import numpy as np, pandas as pd, textwrap
import matplotlib; import matplotlib.pyplot as plt
import vizlib, tslib, geolib, dashlib
from vizlib import PALETTE
vizlib.style_defaults()
pd.set_option("display.width", 200)
CLEAN = Path("clean")
need = ["district_transit_clean.csv", "routes_clean.csv", "ridership_calendar.csv", "fare_revenue_clean.csv"]
assert all((CLEAN / f).exists() for f in need), "Run partA_process.ipynb first: it writes clean/"

reg = pd.read_csv(CLEAN / "district_transit_clean.csv")
rt = pd.read_csv(CLEAN / "routes_clean.csv")
cal = pd.read_csv(CLEAN / "ridership_calendar.csv", parse_dates=["date"])
fr = pd.read_csv(CLEAN / "fare_revenue_clean.csv")
feats = geolib.read_geojson("data/regions.geojson")
parts = fr[~fr.is_total]                                  # the 216 real parts; the 72 subtotals are out
COMPLETE_END = "2026-08-31"                               # 2026-09 is a partial month (Bus, Ferry stopped filing 22 Sep)
MODES = ["Metro", "BRT", "Bus", "Ferry"]
# one hue per mode, everywhere. Ferry's 4th hue is below 3:1 contrast on white, so every mode is ALSO direct-labelled.
MODE_COL = {"Metro": PALETTE["series"][0], "BRT": PALETTE["series"][2], "Bus": PALETTE["series"][1], "Ferry": PALETTE["series_4th"]}
filed = cal[cal.status == "filed"]

def wrap(s, n=160): return "\n".join(textwrap.wrap(s, n))
def frame(fig, title, caption, top=0.84):
    """Finding as the title, the decision it supports as a one-line caption."""
    fig.text(0.01, 0.995, title, ha="left", va="top", fontsize=13.5, fontweight="bold", color=PALETTE["ink"])
    fig.text(0.01, 0.995 - 0.075 * (6.0 / fig.get_size_inches()[1]), caption, ha="left", va="top", fontsize=9.5, style="italic", color=PALETTE["ink_soft"])
    fig.subplots_adjust(top=top)

In [3]:
reg = reg.assign(rate=reg.boardings_annual / reg.population * 1000, stops_per_1000=reg.stops / reg.population * 1000)
r_count = reg[["boardings_annual", "population"]].corr().iloc[0, 1]; r_rate = reg[["rate", "population"]].corr().iloc[0, 1]
top_c = reg.nlargest(3, "boardings_annual").code.tolist(); top_r = reg.nlargest(3, "rate").code.tolist(); both = sorted(set(top_c) & set(top_r))
t5 = reg.nlargest(5, "population"); land5, pop5 = t5.area_km2.sum() / reg.area_km2.sum() * 100, t5.population.sum() / reg.population.sum() * 100
serves = rt.districts_served.str.split(";").explode().value_counts()
r_stops = reg[["rate", "stops_per_1000"]].corr().iloc[0, 1]
pooled = reg.boardings_annual.sum() / reg.population.sum() * 1000
fl = geolib.funnel_limits(reg.population, pooled, per=1000); mm = reg.merge(fl, on="population")
outside = int(((mm.rate < mm.lo) | (mm.rate > mm.hi)).sum()); width = ((fl.hi - fl.lo) / pooled * 100).max()
by = reg.set_index("code")
print(f"r(boardings, population) = {r_count:+.3f} | r(rate, population) = {r_rate:+.3f}")
print("top 3 by count:", top_c, "| top 3 by rate:", top_r, "| on both:", both)
print(f"5 most populous {t5.code.tolist()}: {land5:.1f}% of the land, {pop5:.1f}% of the people")
print("routes serving D09 / D17 / D12 / D21:", [int(serves[c]) for c in ("D09", "D17", "D12", "D21")], "of", len(rt), "| r(rate, stops per 1,000 residents) =", round(r_stops, 3))
print(f"funnel: widest limit is +/-{width/2:.2f}% of the pooled rate; districts OUTSIDE the funnel: {outside} of 24; smallest district count {int(reg.boardings_annual.min()):,}")
assert round(r_count, 3) == 0.945 and round(r_rate, 3) == 0.249 and top_c == ["D17", "D09", "D13"] and top_r == ["D12", "D21", "D09"] and both == ["D09"]
assert round(land5, 1) == 12.6 and round(pop5, 1) == 59.9

r(boardings, population) = +0.945 | r(rate, population) = +0.249
top 3 by count: ['D17', 'D09', 'D13'] | top 3 by rate: ['D12', 'D21', 'D09'] | on both: ['D09']
5 most populous ['D17', 'D09', 'D24', 'D13', 'D11']: 12.6% of the land, 59.9% of the people
routes serving D09 / D17 / D12 / D21: [17, 17, 5, 6] of 36 | r(rate, stops per 1,000 residents) = 0.982
funnel: widest limit is +/-0.07% of the pooled rate; districts OUTSIDE the funnel: 24 of 24; smallest district count 2,391,101


In [4]:
Q = lambda a, b: filed[(filed.date >= a) & (filed.date <= b)]
D_ = dashlib.rate_decomposition(Q("2025-04-01", "2025-06-30"), Q("2026-04-01", "2026-06-30"), "mode", "trips_on_time", "trips_operated")
dec_change, dec_within, dec_mix = D_["change"], D_["within"], D_["mix"]
lost = [c for c in top_r if c not in top_c]; gained = [c for c in top_c if c not in top_r]
nm = by.district.to_dict(); P = lambda c: f"{by.population[c]/1e6:.1f}M"
note = f"""# Task E - the note

**Message (one sentence).** The Authority's "punctuality is improving" is a mix effect (every mode got worse: the {dec_change:+.2f}-pt headline = {dec_within:+.2f} within modes + {dec_mix:+.2f} mix), and the next corridor should be argued from boardings *per resident*, not boardings.
**Reader.** The Secretary (30 seconds, top-left only) first; the finance director and the journalist second.

## Parameters, each with a reason
- **Window:** 2025 Q2 vs 2026 Q2, the latest complete quarters a year apart (the reversal also holds for first-3/last-3 and first-6/last-6 months); partial 2026-09 is never compared.
- **Unit:** departures for punctuality, boardings (legs) for map and trend, journeys only for fares; never added or on one axis (boardings / journeys = 1.30).
- **Aggregation:** sum on-time and operated, *then* divide (a mean of daily % weights a quiet Sunday like a busy Tuesday); unfiled days break lines, never zeros.
- **Denominator:** resident population from the register, in words on the legend ("per 1,000 residents").
- **Breaks:** Jenks natural breaks, k=5, edges printed; equal breaks would put one outlier in its own class.
- **Projection:** equal-area, because colouring an area makes area the encoding.

## The counterfactual
A count map of boardings is a population map: r(boardings, population) = **{r_count:+.3f}**, against **{r_rate:+.3f}** for the rate. Raw counts send the corridor to **{', '.join(top_c)}** ({', '.join(P(c) for c in top_c)} residents); the rate points at **{', '.join(top_r)}**. Only **{both[0]}** ({nm[both[0]]}) is on both lists, and *why* is the finding: it is the network hub - {int(serves['D09'])} of 36 routes serve it (tied for most) and it has the most stops - so it is large *and* intense. The count version takes the corridor away from **{' and '.join(f'{c} {nm[c]} ({P(c)})' for c in lost)}**, the two small districts with the highest boardings per resident, and gives it to {' and '.join(f'{c} {nm[c]}' for c in gained)}. The five most populous districts hold {pop5:.1f}% of the people on {land5:.1f}% of the land, which is all a count map can see.

## Ethics clause
**Suppressed or merged: none (0 of 24).** My rule would have been to suppress a rate whose Poisson funnel interval contains the pooled rate; the funnel is only +/-{width/2:.2f}% wide (smallest district count {int(reg.boardings_annual.min()):,} boardings) and **{outside} of 24** districts lie outside it, so noise is not what separates them - structure is. Three districts (D05, D15, D23) were filed in thousands and are *repaired*, not suppressed; the route table agrees with the repaired register to within 690 boardings. **What I would not claim:** a high rate is not "well served". Boardings are credited to where a route runs, so a hub is credited with non-residents' trips, and the rate tracks stop density almost exactly (r = {r_stops:+.2f}). Siting infrastructure from this screen should add access (stops, income, concession share), not only rank use; the data are synthetic and the district names carry no real-world meaning.
"""
Path("partE_note.md").write_text(note, encoding="utf-8"); print(note); print("\nwords:", len(note.split()))

# Task E - the note

**Message (one sentence).** The Authority's "punctuality is improving" is a mix effect (every mode got worse: the +1.73-pt headline = -1.10 within modes + +2.82 mix), and the next corridor should be argued from boardings *per resident*, not boardings.
**Reader.** The Secretary (30 seconds, top-left only) first; the finance director and the journalist second.

## Parameters, each with a reason
- **Window:** 2025 Q2 vs 2026 Q2, the latest complete quarters a year apart (the reversal also holds for first-3/last-3 and first-6/last-6 months); partial 2026-09 is never compared.
- **Unit:** departures for punctuality, boardings (legs) for map and trend, journeys only for fares; never added or on one axis (boardings / journeys = 1.30).
- **Aggregation:** sum on-time and operated, *then* divide (a mean of daily % weights a quiet Sunday like a busy Tuesday); unfiled days break lines, never zeros.
- **Denominator:** resident population from the register, in words on the legen